# Evaluation with DeepEval

Score a RAG pipeline with DeepEval's LLM-as-judge metrics, using a local Ollama model as the judge.

**Extra requirements:** `pip install deepeval`

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric, ContextualRelevancyMetric
from deepeval.test_case import LLMTestCase

class LocalJudge(DeepEvalBaseLLM):
    def load_model(self): return None
    def generate(self, prompt: str) -> str: return llm(prompt)
    async def a_generate(self, prompt: str) -> str: return llm(prompt)
    def get_model_name(self): return LLM_MODEL

judge = LocalJudge()
metrics = [
    AnswerRelevancyMetric(model=judge, threshold=0.7),
    FaithfulnessMetric(model=judge, threshold=0.7),
    ContextualRelevancyMetric(model=judge, threshold=0.5),
]

Build one test case per golden question: the question, our answer, and the retrieved context.

In [ ]:
cases = []
for q in load_eval_questions()[:4]:
    response, hits = rag(q["question"], store, k=3)
    cases.append(LLMTestCase(input=q["question"], actual_output=response,
                             expected_output=q["answer"], retrieval_context=[h["text"] for h in hits]))

In [ ]:
for case in cases:
    print("Q:", case.input)
    for metric in metrics:
        metric.measure(case)
        print(f"  {type(metric).__name__:28s} {metric.score:.2f}  {'PASS' if metric.is_successful() else 'FAIL'}")

Small local judges are noisy - run each metric several times or use a stronger judge for decisions that matter.